# 02 RAG知识库构建与检索

**用途：** 亲手观察Document、Splitter、Embedding配置、VectorStore和带距离的Retriever。

> 使用方式：按顺序运行。出现 `PASS` 才代表本节验收成功；断言失败时先阅读紧邻的“失败定位”。默认不调用真实模型、不写生产数据库。

In [1]:
from pathlib import Path
import importlib.util
import json
import os
import sys
import tempfile

cwd = Path.cwd().resolve()
DAY1_ROOT = None
PROJECT2_ROOT = None
for candidate in [cwd, *cwd.parents]:
    if (candidate / "project2" / "agent_graph.py").exists():
        DAY1_ROOT = candidate
        PROJECT2_ROOT = candidate / "project2"
        break
    if (candidate / "agent_graph.py").exists() and (candidate / "tests").exists():
        PROJECT2_ROOT = candidate
        DAY1_ROOT = candidate.parent
        break
assert DAY1_ROOT is not None and PROJECT2_ROOT is not None, "找不到 day1/project2 项目根目录"
NOTEBOOK_ROOT = PROJECT2_ROOT / "notebooks"
for path in [str(DAY1_ROOT), str(PROJECT2_ROOT), str(NOTEBOOK_ROOT)]:
    if path not in sys.path:
        sys.path.insert(0, path)

from notebook_utils import (
    check,
    check_equal,
    file_inventory,
    load_jsonl,
    masked_environment,
    run_command,
    run_unittest,
    show_markdown,
    show_table,
    source_excerpt,
)

RUN_LIVE_MODEL_TESTS = os.getenv("RUN_LIVE_MODEL_TESTS", "0") == "1"
print(f"Python: {sys.executable}")
print(f"DAY1_ROOT: {DAY1_ROOT}")
print(f"PROJECT2_ROOT: {PROJECT2_ROOT}")
print(f"RUN_LIVE_MODEL_TESTS: {RUN_LIVE_MODEL_TESTS}")

Python: D:\new things\项目1\day1\.venv\Scripts\python.exe
DAY1_ROOT: D:\new things\项目1\day1
PROJECT2_ROOT: D:\new things\项目1\day1\project2
RUN_LIVE_MODEL_TESTS: False


## 1. 当前RAG链路

```text
文件 -> Document -> RecursiveCharacterTextSplitter
    -> bge-small-zh-v1.5 Embedding -> Chroma
    -> Scored Retriever -> Prompt -> DeepSeek -> 来源与工单
```

当前默认 `chunk_size=500`、`overlap=80`。这不是通用真理，而是根据中文客服资料段落长度、字段完整性和评测成本选出的起点。

In [2]:
from settings import (
    EMBEDDING_MODEL, EMBEDDING_DEVICE, EMBEDDING_NORMALIZE,
    RAG_CHUNK_SIZE, RAG_CHUNK_OVERLAP, RAG_MAX_DISTANCE,
    VECTOR_DB_PROVIDER, VECTOR_DB_DISTANCE_METRIC,
)
config = [
    {"配置": "embedding", "值": EMBEDDING_MODEL},
    {"配置": "device", "值": EMBEDDING_DEVICE},
    {"配置": "normalize", "值": EMBEDDING_NORMALIZE},
    {"配置": "chunk_size", "值": RAG_CHUNK_SIZE},
    {"配置": "chunk_overlap", "值": RAG_CHUNK_OVERLAP},
    {"配置": "max_distance", "值": RAG_MAX_DISTANCE},
    {"配置": "vector_db", "值": VECTOR_DB_PROVIDER},
    {"配置": "distance_metric", "值": VECTOR_DB_DISTANCE_METRIC},
]
show_table(config)
check_equal("默认chunk size", RAG_CHUNK_SIZE, 500)
check_equal("默认overlap", RAG_CHUNK_OVERLAP, 80)

,配置,值
0,embedding,BAAI/bge-small-zh-v1.5
1,device,cpu
2,normalize,True
3,chunk_size,500
4,chunk_overlap,80
5,max_distance,1.0
6,vector_db,chroma
7,distance_metric,l2


[PASS] 默认chunk size | actual=500, expected=500
[PASS] 默认overlap | actual=80, expected=80


{'检查项': '默认overlap', '状态': 'PASS', '说明': 'actual=80, expected=80'}

In [3]:
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

sample = ("PC200液压泵适配前必须核对设备铭牌、旧件号和出厂年份。"
          "库存与价格必须调用工具查询，不能根据知识库编造。" * 25)
source_doc = Document(page_content=sample, metadata={"source": "notebook_demo.md"})
splitter = RecursiveCharacterTextSplitter(
    chunk_size=RAG_CHUNK_SIZE,
    chunk_overlap=RAG_CHUNK_OVERLAP,
)
chunks = splitter.split_documents([source_doc])
show_table([
    {"序号": i + 1, "字符数": len(item.page_content), "来源": item.metadata["source"],
     "开头": item.page_content[:45]}
    for i, item in enumerate(chunks)
])
check("文档被切成多个chunk", len(chunks) > 1)
check("chunk没有超过配置", max(len(item.page_content) for item in chunks) <= RAG_CHUNK_SIZE)

D:\new things\项目1\day1\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


,序号,字符数,来源,开头
0,1,500,notebook_demo.md,PC200液压泵适配前必须核对设备铭牌、旧件号和出厂年份。库存与价格必须调用工具查询，不能
1,2,500,notebook_demo.md,库编造。PC200液压泵适配前必须核对设备铭牌、旧件号和出厂年份。库存与价格必须调用工具查
2,3,485,notebook_demo.md,根据知识库编造。PC200液压泵适配前必须核对设备铭牌、旧件号和出厂年份。库存与价格必须调


[PASS] 文档被切成多个chunk
[PASS] chunk没有超过配置


{'检查项': 'chunk没有超过配置', '状态': 'PASS', '说明': ''}

In [4]:
from rag_components import create_retriever

class FakeVectorStore:
    def similarity_search_with_score(self, query, k):
        return [
            (Document(page_content=f"{query}需要核对旧件号", metadata={"source": "docs/fit.md"}), 0.25),
            (Document(page_content="报价由工具生成", metadata={"source": "docs/quote.md"}), 0.72),
        ][:k]

retriever = create_retriever(FakeVectorStore(), k=3)
retrieved = retriever.invoke("PC200液压泵适配")
show_table([
    {
        "rank": doc.metadata["retrieval_rank"],
        "distance": doc.metadata["retrieval_distance"],
        "provider": doc.metadata["retrieval_provider"],
        "source": doc.metadata["source"],
        "content": doc.page_content,
    }
    for doc in retrieved
])
check_equal("检索结果数量", len(retrieved), 2)
check_equal("第一条rank", retrieved[0].metadata["retrieval_rank"], 1)

,rank,distance,provider,source,content
0,1,0.25,chroma,docs/fit.md,PC200液压泵适配需要核对旧件号
1,2,0.72,chroma,docs/quote.md,报价由工具生成


[PASS] 检索结果数量 | actual=2, expected=2
[PASS] 第一条rank | actual=1, expected=1


{'检查项': '第一条rank', '状态': 'PASS', '说明': 'actual=1, expected=1'}

In [5]:
rag_tests = run_unittest(["tests.test_rag_components"], project2_root=DAY1_ROOT)
check("RAG组件4条测试通过", "Ran 4 tests" in rag_tests.output and "OK" in rag_tests.output)

$ D:\new things\项目1\day1\.venv\Scripts\python.exe -m unittest tests.test_rag_components -v
test_fingerprint_changes_detects_embedding_or_chunk_change (tests.test_rag_components.RagComponentTests.test_fingerprint_changes_detects_embedding_or_chunk_change) ... ok
test_index_fingerprint_records_vector_compatibility (tests.test_rag_components.RagComponentTests.test_index_fingerprint_records_vector_compatibility) ... ok
test_rag_generation_is_a_langchain_runnable (tests.test_rag_components.RagComponentTests.test_rag_generation_is_a_langchain_runnable) ... ok
test_scored_retriever_preserves_rank_distance_and_provider (tests.test_rag_components.RagComponentTests.test_scored_retriever_preserves_rank_distance_and_provider) ... ok

----------------------------------------------------------------------
Ran 4 tests in 0.008s

OK
[PASS] RAG组件4条测试通过


{'检查项': 'RAG组件4条测试通过', '状态': 'PASS', '说明': ''}

## 关键问题与答案

- **为什么500/80？** 500字符通常能保留一个中文业务规则，80字符缓解边界截断；最终要靠检索命中、来源正确率、噪声和Token成本评测。
- **为什么bge-small-zh-v1.5？** 中文语义效果、CPU可运行、体积和延迟比较平衡；限制是专业件号和表格数字仍需关键词、元数据或混合检索补充。
- **Document、VectorStore、Retriever关系？** Document是内容和元数据；VectorStore保存向量并搜索；Retriever把搜索封装成稳定接口并补rank、distance、provider。
- **为什么不直接手写向量检索？** LangChain降低组件替换成本，但代价是抽象层、版本变化和调试复杂度，所以距离阈值与业务判断仍由项目控制。

### 面试代码追问

1. `create_retriever()`如何保留距离和来源？
2. 更换Embedding为什么必须重建索引？
3. overlap过大会造成什么重复和Token问题？
4. 件号严格匹配为什么不能只依赖语义Embedding？

### 参考答案

1. **`create_retriever()`如何保留距离和来源？** 它调用`similarity_search_with_score`，再把返回的score写入每个Document的`retrieval_distance`，同时补充`retrieval_rank`和`retrieval_provider`，原始`source`元数据继续保留，所以上层既能生成答案，也能做阈值、引用和评测。
2. **为什么换Embedding必须重建索引？** 向量库中已有向量是旧模型坐标空间的结果。新模型的维度和空间分布可能不同，查询向量不能与旧文档向量直接比较；项目用index fingerprint检测这种不兼容变化。
3. **overlap过大会怎样？** 同一句话会进入多个chunk，导致重复召回、上下文噪声、Token和存储增加，还可能让来源命中看似变高。overlap过小则容易切断跨边界规则，所以要结合召回和成本评测。
4. **为什么件号不能只靠语义Embedding？** 件号是高精度标识符，字符差一位可能就是不同零件，而Embedding可能把相似字符串映射得很近。应增加规范化、关键词/BM25、元数据过滤或精确匹配，再用语义检索补自然语言表达。

**代码落点：** `rag_components.py::create_retriever`、`build_index_fingerprint`，以及`settings.py`中的切分和距离配置。